In [1]:
# Use this initial code to work in the notebook as if it were a module, that
# is, to be able to export classes and functions from other subpackages.

import os
import sys

package_path = os.path.abspath(".").split(os.sep + "notebooks")[0]
if package_path not in sys.path:
    sys.path.append(package_path)

%load_ext autoreload
%autoreload 2

In [3]:
import urllib, urllib.request
url = 'http://export.arxiv.org/api/query?search_query=all:electron&start=0&max_results=1'
data = urllib.request.urlopen(url)
print(data.read().decode('utf-8'))


<?xml version='1.0' encoding='UTF-8'?>
<feed xmlns:opensearch="http://a9.com/-/spec/opensearch/1.1/" xmlns:arxiv="http://arxiv.org/schemas/atom" xmlns="http://www.w3.org/2005/Atom">
  <id>https://arxiv.org/api/cHxbiOdZaP56ODnBPIenZhzg5f8</id>
  <title>arXiv Query: search_query=all:electron&amp;id_list=&amp;start=0&amp;max_results=1</title>
  <updated>2026-04-13T16:31:38Z</updated>
  <link href="https://arxiv.org/api/query?search_query=all:electron&amp;start=0&amp;max_results=1&amp;id_list=" type="application/atom+xml"/>
  <opensearch:itemsPerPage>1</opensearch:itemsPerPage>
  <opensearch:totalResults>180125</opensearch:totalResults>
  <opensearch:startIndex>0</opensearch:startIndex>
  <entry>
    <id>http://arxiv.org/abs/cond-mat/0011267v1</id>
    <title>The electronic structure of cuprates from high energy spectroscopy</title>
    <updated>2000-11-15T16:19:15Z</updated>
    <link href="https://arxiv.org/abs/cond-mat/0011267v1" rel="alternate" type="text/html"/>
    <link href="https:

In [24]:
import httpx
import xml.etree.ElementTree as ET
from src.researchos.domain.models import Paper

BASE_URL = 'https://export.arxiv.org/api/query'
NS = {
        "atom": "http://www.w3.org/2005/Atom",
        "arxiv": "http://arxiv.org/schemas/atom",
    }

async def search_papers(query: str, max_results: int) -> list[Paper]:
    params = {
        "search_query": query,
        "start": 0,
        "max_results": max_results
    }

    async with httpx.AsyncClient() as client:
        response = await client.get(BASE_URL, params=params)
        return _parse_entries(response)

def _parse_entries(results: httpx.Response) -> list[Paper]:

    root = ET.fromstring(results.text)

    papers = []

    for entry in root.findall("atom:entry", NS):
        id = entry.findtext("atom:id", namespaces=NS)
        title = entry.findtext("atom:title", namespaces=NS)
        summary = entry.findtext("atom:summary", namespaces=NS)
        published = entry.findtext("atom:published", namespaces=NS)
        authors = [author.findtext("atom:name", namespaces=NS) for author in entry.findall("atom:author", NS)]
        categories = [cat.get('term') for cat in entry.findall("atom:category", NS)]
        for link in entry.findall("atom:link", NS):
            if link.get('title') == 'pdf': 
                pdf_url = link.get('href')
            elif link.get("rel") == "alternate":
                url = link.get("href")
            

        paper = Paper(
            source_id=id,
            source="arxiv",
            title=title,
            abstract=summary,
            authors=authors,
            published_date=published,
            url=url,
            pdf_url=pdf_url,
            categories=categories
        )
        papers.append(paper)

    return papers


q= "electron"
m = 1
results = await search_papers(query=q, max_results=m)

results

[Paper(source_id='http://arxiv.org/abs/cond-mat/0011267v1', source='arxiv', title='The electronic structure of cuprates from high energy spectroscopy', authors=['Mark S. Golden', 'Christian Duerr', 'Andreas Koitzsch', 'Sibylle Legner', 'Zhiwei Hu', 'Sergey Borisenko', 'Martin Knupfer', 'Joerg Fink'], abstract='  We report studies of the electronic structure and elementary excitations of doped and undoped cuprate chains, ladders and planes. Using high energy spectroscopies such as x-ray absorption, core level photoemission and angle resolved photoemission spectroscopy, important information regarding the charge distribution and hole dynamics can be obtained. The comparison of the experimental data with suitable theoretical models sets constraints on the parameters entering into the model calculations, and offers insight into the important physical quantities governing the electronic structure of these materials. Recurring themes include the importance of the dimensionality of the Cu-O n

In [19]:
fields

[Paper(source_id='http://arxiv.org/abs/cond-mat/0011267v1', source='arxiv', title='The electronic structure of cuprates from high energy spectroscopy', authors=['Mark S. Golden', 'Christian Duerr', 'Andreas Koitzsch', 'Sibylle Legner', 'Zhiwei Hu', 'Sergey Borisenko', 'Martin Knupfer', 'Joerg Fink'], abstract='  We report studies of the electronic structure and elementary excitations of doped and undoped cuprate chains, ladders and planes. Using high energy spectroscopies such as x-ray absorption, core level photoemission and angle resolved photoemission spectroscopy, important information regarding the charge distribution and hole dynamics can be obtained. The comparison of the experimental data with suitable theoretical models sets constraints on the parameters entering into the model calculations, and offers insight into the important physical quantities governing the electronic structure of these materials. Recurring themes include the importance of the dimensionality of the Cu-O n

# Prueba desde módulo

In [3]:
from src.researchos.infrastructure.data.arxiv import search_papers

q= "electron"
m = 1
results = await search_papers(query=q, max_results=m)

results

[Paper(source_id='http://arxiv.org/abs/cond-mat/0011267v1', source='arxiv', title='The electronic structure of cuprates from high energy spectroscopy', authors=['Mark S. Golden', 'Christian Duerr', 'Andreas Koitzsch', 'Sibylle Legner', 'Zhiwei Hu', 'Sergey Borisenko', 'Martin Knupfer', 'Joerg Fink'], abstract='  We report studies of the electronic structure and elementary excitations of doped and undoped cuprate chains, ladders and planes. Using high energy spectroscopies such as x-ray absorption, core level photoemission and angle resolved photoemission spectroscopy, important information regarding the charge distribution and hole dynamics can be obtained. The comparison of the experimental data with suitable theoretical models sets constraints on the parameters entering into the model calculations, and offers insight into the important physical quantities governing the electronic structure of these materials. Recurring themes include the importance of the dimensionality of the Cu-O n

In [5]:
q= "electron"
m = 2
results = await search_papers(query=q, max_results=m)

results

[Paper(source_id='http://arxiv.org/abs/cond-mat/0011267v1', source='arxiv', title='The electronic structure of cuprates from high energy spectroscopy', authors=['Mark S. Golden', 'Christian Duerr', 'Andreas Koitzsch', 'Sibylle Legner', 'Zhiwei Hu', 'Sergey Borisenko', 'Martin Knupfer', 'Joerg Fink'], abstract='  We report studies of the electronic structure and elementary excitations of doped and undoped cuprate chains, ladders and planes. Using high energy spectroscopies such as x-ray absorption, core level photoemission and angle resolved photoemission spectroscopy, important information regarding the charge distribution and hole dynamics can be obtained. The comparison of the experimental data with suitable theoretical models sets constraints on the parameters entering into the model calculations, and offers insight into the important physical quantities governing the electronic structure of these materials. Recurring themes include the importance of the dimensionality of the Cu-O n

In [7]:
q= "LLM agents"
m = 3
results = await search_papers(query=q, max_results=m)

results

[Paper(source_id='http://arxiv.org/abs/2604.08224v1', source='arxiv', title='Externalization in LLM Agents: A Unified Review of Memory, Skills, Protocols and Harness Engineering', authors=['Chenyu Zhou', 'Huacan Chai', 'Wenteng Chen', 'Zihan Guo', 'Rong Shan', 'Yuanyi Song', 'Tianyi Xu', 'Yingxuan Yang', 'Aofan Yu', 'Weiming Zhang', 'Congming Zheng', 'Jiachen Zhu', 'Zeyu Zheng', 'Zhuosheng Zhang', 'Xingyu Lou', 'Changwang Zhang', 'Zhihui Fu', 'Jun Wang', 'Weiwen Liu', 'Jianghao Lin', 'Weinan Zhang'], abstract='Large language model (LLM) agents are increasingly built less by changing model weights than by reorganizing the runtime around them. Capabilities that earlier systems expected the model to recover internally are now externalized into memory stores, reusable skills, interaction protocols, and the surrounding harness that makes these modules reliable in practice. This paper reviews that shift through the lens of externalization. Drawing on the idea of cognitive artifacts, we argue